# EnergyCast V1 — Step 9: Seasonal Naive Baseline Models
## Electricity Price Forecasting System (Indian Electricity Market)

**Objective:** Establish non-parametric time-series benchmarks—Daily Seasonal Naive ($s=24$), Weekly Seasonal Naive ($s=168$), and Simple Persistence ($s=1$)—across Validation (2024) and Test (2025) splits.  
These benchmarks establish the fundamental hurdle that Linear/Ridge (Step 10) and Gradient Boosted Trees (Step 11) must surpass to demonstrate genuine algorithmic learning.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set visual style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)

# Paths
METRICS_PATH = Path('../outputs/tables/seasonal_naive_metrics.csv')
PREDS_PATH = Path('../outputs/predictions/snaive_test_predictions.csv')
print('Metrics CSV exists:', METRICS_PATH.exists())
print('Test Predictions CSV exists:', PREDS_PATH.exists())

### 1. Benchmark Performance Metrics Table
We inspect the empirical MAE, RMSE, sMAPE, and R2 across Validation (2024) and Test (2025).

In [ ]:
metrics_df = pd.read_csv(METRICS_PATH)
display(metrics_df)

### 2. Time-Series Forecast Tracking Sample (14-Day Test Horizon)
Visualizing actual ground truth MCP against Daily Seasonal Naive ($y_{t-24}$) and Weekly Seasonal Naive ($y_{t-168}$).

In [ ]:
preds_df = pd.read_csv(PREDS_PATH)
preds_df['timestamp'] = pd.to_datetime(preds_df['timestamp'])

sample = preds_df.iloc[1000:1336] # 14-day sample
plt.figure(figsize=(15, 6))
plt.plot(sample['timestamp'], sample['actual_mcp'], label='Actual MCP ($y_t$)', color='#1F4E79', lw=2)
plt.plot(sample['timestamp'], sample['pred_daily_snaive_24h'], label='Daily sNaive ($y_{t-24}$)', color='#E67E22', ls='--', lw=1.5)
plt.plot(sample['timestamp'], sample['pred_weekly_snaive_168h'], label='Weekly sNaive ($y_{t-168}$)', color='#D9534F', ls=':', lw=1.2)

plt.title('Seasonal Naive Benchmark Tracking vs Ground Truth MCP (Test Set)', fontweight='bold')
plt.ylabel('Price (Rs/MWh)')
plt.xlabel('Date')
plt.legend(loc='upper right')
plt.show()

### 3. Hourly MAE Profile (Peak-Hour Vulnerability Analysis)
Where does Seasonal Naive struggle? Examining MAE across each hour of the day reveals massive error spikes during the 18:00 to 22:00 evening peak.

In [ ]:
preds_df['hour'] = preds_df['timestamp'].dt.hour
preds_df['err_24'] = np.abs(preds_df['actual_mcp'] - preds_df['pred_daily_snaive_24h'])
preds_df['err_168'] = np.abs(preds_df['actual_mcp'] - preds_df['pred_weekly_snaive_168h'])

hourly_mae = preds_df.groupby('hour')[['err_24', 'err_168']].mean()

plt.figure(figsize=(12, 5.5))
plt.plot(hourly_mae.index, hourly_mae['err_24'], marker='o', color='#E67E22', lw=2, label='Daily sNaive (24h) Hourly MAE')
plt.plot(hourly_mae.index, hourly_mae['err_168'], marker='s', color='#D9534F', lw=2, label='Weekly sNaive (168h) Hourly MAE')
plt.axvspan(18, 22, color='red', alpha=0.1, label='Evening Peak (18:00 - 22:00)')
plt.title('Hourly MAE Profile: Where Naive Baselines Fail', fontweight='bold')
plt.xlabel('Hour of Day (0 to 23)')
plt.ylabel('Mean Absolute Error (Rs/MWh)')
plt.xticks(range(0, 24, 2))
plt.legend()
plt.show()

### 4. Summary & Benchmark Hurdle for Machine Learning
The best baseline is **Daily Seasonal Naive ($s=24$)**:
- **Validation MAE:** ₹629.84 / MWh | **RMSE:** ₹1,185.06 / MWh | **sMAPE:** 14.97% | **R²:** 0.7918
- **Test MAE:** ₹658.42 / MWh | **RMSE:** ₹1,254.07 / MWh | **sMAPE:** 16.52% | **R²:** 0.7828

Any model in **Step 10 (Linear/Ridge)** and **Step 11 (LightGBM/XGBoost)** must beat these thresholds.